# IBVAP — coarse detector training on Colab

13-class YOLO11s, leak-free 89k/10k split. Built for **Colab Pro** (Google AI Pro).

**Before you start:** upload `data/ibvap_coarse_dataset.tar` (~33 GB, built by
`scripts/pack_dataset_for_colab.py --tar`) to `MyDrive/ibvap/` in your Drive.

Runtime → Change runtime type → **A100** (or L4). Then Runtime → Run all.

Weights are written **straight to Drive every epoch**, so a disconnect loses
nothing. To continue: new runtime → run cells 1–3 (GPU, mount, extract) →
run the **Resume** cell (not the fresh-train cell).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install 'ultralytics==8.4.140' 'torch' 2>/dev/null
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE   = '/content/drive/MyDrive/ibvap'
TAR     = f'{DRIVE}/ibvap_coarse_dataset.tar'
DATA    = '/content/ibvap_data'          # fast local disk
RUNS    = f'{DRIVE}/runs'                # checkpoints live here (survive disconnects)
RUN     = 'ibvap_coarse'
os.makedirs(RUNS, exist_ok=True)
assert os.path.exists(TAR), f'not found: {TAR} — upload the tar to MyDrive/ibvap/ first'
print('tar size: %.1f GB' % (os.path.getsize(TAR)/1e9))

In [ ]:
# Extract once to local disk (~15–25 min). Skips if already done.
import os, subprocess, time
if not os.path.isdir(f'{DATA}/images'):
    os.makedirs(DATA, exist_ok=True)
    t = time.time()
    subprocess.run(['tar', '-xf', TAR, '-C', DATA], check=True)
    print('extracted in %.1f min' % ((time.time()-t)/60))
else:
    print('already extracted')
!ls {DATA} && head -c 200 {DATA}/dataset_coarse.yaml

In [ ]:
# Weights go straight to Drive every epoch — a disconnect can't lose them.
import os
from ultralytics import YOLO, settings

# dataset on fast local disk; run outputs (weights, results.csv, plots) on Drive
settings.update({'datasets_dir': DATA, 'runs_dir': RUNS})

CKPT = f'{RUNS}/{RUN}'
os.makedirs(f'{CKPT}/weights', exist_ok=True)
print('run dir on Drive:', CKPT)
print('  last.pt ->', f'{CKPT}/weights/last.pt')


In [ ]:
import re

# Point the dataset yaml at the local copy (idempotent — safe to re-run).
yaml_path = f'{DATA}/dataset_coarse.yaml'
with open(yaml_path) as f:
    y = f.read()
y = re.sub(r'(?m)^path:.*$', f'path: {DATA}', y)
with open(yaml_path, 'w') as f:
    f.write(y)
print(y.splitlines()[0])

# ---- TRAIN (fresh) ----
# Run this ONCE. After a disconnect use the Resume cell below instead.
TRAIN_ARGS = dict(
    data=yaml_path,
    epochs=150, imgsz=640, batch=-1,          # A100: try batch=64 explicitly if -1 underfills
    patience=40, device=0, workers=8, cache='disk',
    optimizer='SGD', lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005,
    warmup_epochs=3.0, cos_lr=True, close_mosaic=15,
    mixup=0.10, copy_paste=0.15, degrees=5.0, scale=0.5,
    project=RUNS, name=RUN, exist_ok=True, plots=True,
)
model = YOLO('yolo11s.pt')
model.train(**TRAIN_ARGS)


In [ ]:
# ================== RESUME after a disconnect ==================
# New runtime? First run cell 1 (GPU/install), cell 2 (mount Drive), and
# cell 3 (re-extract dataset to local disk). Then run this cell.
import os, re
from ultralytics import YOLO

DRIVE = '/content/drive/MyDrive/ibvap'
DATA  = '/content/ibvap_data'
RUNS  = f'{DRIVE}/runs'
RUN   = 'ibvap_coarse'

run_dir = f'{RUNS}/{RUN}'
last = f'{run_dir}/weights/last.pt'
if not os.path.exists(last):
    if os.path.isdir(run_dir):
        raise FileNotFoundError(f'{last} missing. run dir holds: ' + ', '.join(sorted(os.listdir(run_dir))))
    raise FileNotFoundError(f'{run_dir} does not exist - is Drive mounted (cell 2)? did the fresh run reach epoch 1?')

# the checkpoint stores data=<this path>; it must exist and point at the local copy
yaml_path = f'{DATA}/dataset_coarse.yaml'
assert os.path.exists(yaml_path), f'run the extract cell (cell 3) first - {yaml_path} missing'
y = open(yaml_path).read()
open(yaml_path, 'w').write(re.sub(r'(?m)^path:.*$', f'path: {DATA}', y))

print('resuming from', last)
model = YOLO(last)
model.train(resume=True)


In [ ]:
# ---- Validate best.pt + copy it out of the run dir ----
import shutil
best = f'{RUNS}/{RUN}/weights/best.pt'
m = YOLO(best)
metrics = m.val(data=f'{DATA}/dataset_coarse.yaml', imgsz=640, split='val')
print('mAP50', round(metrics.box.map50, 3), '| mAP50-95', round(metrics.box.map, 3))
shutil.copy2(best, f'{DRIVE}/ibvap_coarse_best.pt')
print('saved -> MyDrive/ibvap/ibvap_coarse_best.pt  (download this into backend/models/)')
